# Creación de un modelo TiDE con Darts

Como hemos visto en notebooks anteriores, un modelo muy útil cuando tratamos con varios gigas de datos es el TiDE. Anteriormente lo entrenamos con pytorch-forecasting, pero a la hora de llevarlo a producción es una libreria muy tediosa. Como respuesta Darts ofrece clases más amigables y sencillas de entender, a cambio de un tuning más simple y menos ajustable.

Toda la información del modelo ha sido extraida de la web oficial de Darts https://unit8co.github.io/darts/generated_api/darts.models.forecasting.tide_model.html#module-darts.models.forecasting.tide_model

Descarguemos los datos y hagamos las transformaciones necesarias

In [20]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

root = Path.cwd().parent 
sys.path.append(str(root))

from minio_utils import MinioSparkClient


from pyspark.ml.feature import StringIndexer, OneHotEncoder,VectorAssembler, StandardScaler, SQLTransformer, Imputer
from pyspark.sql.types import FloatType
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsWritable, DefaultParamsReadable
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
import math
from torch.utils.data import IterableDataset, DataLoader
import pyarrow.parquet as pq

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import TiDEModel
from darts.dataprocessing.transformers import MissingValuesFiller
from darts.utils.likelihood_models import QuantileRegression

from clearml import Task
from tqdm.auto import tqdm

from setup import setenv
setenv()


In [2]:

spark = MinioSparkClient(
    endpoint=os.getenv("MINIO_ENDPOINT", "").replace("http://", "").replace("https://", ""),
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    bucket_name="pd2",
    base_dir="cityenjoyer",
    memory = 16,
    heapsize = 8,
    num_part = 2000,
    verbose=True
)
spark.connect()

26/03/25 10:21:33 WARN Utils: Your hostname, danpanto-OMEN-Gaming-Laptop-16-ap0xxx resolves to a loopback address: 127.0.1.1; using 10.8.56.66 instead (on interface wlo1)
26/03/25 10:21:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://mmlspark.azureedge.net/maven added as a remote repository with the name: repo-1
Ivy Default Cache set to: /home/danpanto/.ivy2/cache
The jars for the packages stored in: /home/danpanto/.ivy2/jars
com.microsoft.azure#synapseml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-03efc7ef-c8e9-4b54-8109-73a76dbb6432;1.0
	confs: [default]
	found com.microsoft.azure#synapseml_2.12;1.1.2 in central


:: loading settings :: url = jar:file:/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.microsoft.azure#synapseml-core_2.12;1.1.2 in central
	found org.apache.spark#spark-avro_2.12;3.5.0 in central
	found org.tukaani#xz;1.9 in central
	found commons-lang#commons-lang;2.6 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central
	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found com.linkedin.isolation-forest#isolation-forest_3.5.

In [3]:

df = spark.read_parquet("prepared_for_model/20260323_132105_agg.parquet")

df.show()

+--------+------------+-------------------+------+------------+----------+---------+----------+----+---+--------------------+--------------------+-------------------+-------------------+
|VendorID|PULocationID|          timestamp|demand|avg_distance|avg_amount| Latitude| Longitude|hour|dow|            hour_sin|            hour_cos|            dow_sin|            dow_cos|
+--------+------------+-------------------+------+------------+----------+---------+----------+----+---+--------------------+--------------------+-------------------+-------------------+
|       0|         263|2023-06-01 04:00:00|    14|     10730.5| 3877.7144|40.778767| -73.95101|   4|  5|  0.8660254037844386|  0.5000000000000001|-0.9749279121818236|-0.2225209339563146|
|       0|         262|2023-06-01 06:00:00|    59|    4567.576| 2454.5425|40.775932| -73.94651|   6|  5|                 1.0|6.123233995736766...|-0.9749279121818236|-0.2225209339563146|
|       0|         163|2023-06-01 09:00:00|   141|   4182.0566| 2

Para usar darts ya no necesitamos usar onehotencoders o los centroides, la propia libreria nos proporciona embeddings que van a aprender los mejores vectores para cada zona

In [4]:
# Limpiamos nulos
df_clean = df.na.drop(subset=["demand", "PULocationID", "VendorID", "timestamp"])

# Creamos una columna que sea el ID único de la serie temporal
df_clean = df_clean.withColumn(
    "Series_ID", 
    F.concat_ws("_", F.col("VendorID"), F.col("PULocationID"))
)

# Ahora particionamos por esa nueva columna, 
# esto nos permite tener un folder por cada combinación y 
# nos permite usar darts con mini datasets que podemos pasar a pandas
df_clean.write \
    .partitionBy("Series_ID") \
    .mode("overwrite") \
    .parquet("../data/data_darts.parquet")

print("Datos exportados correctamente por Vendor y Zona.")

Datos exportados correctamente por Vendor y Zona.


Ahora creamos las listas preparadas para usar darts

In [7]:
base_path = "../data/data_darts.parquet/"
# Buscamos las carpetas con la nueva clave (ej: Series_ID=0_262)
series_folders = [d for d in os.listdir(base_path) if d.startswith("Series_ID=")]

target_series_list = []
past_covariates_list = []
future_covariates_list = []

for folder in series_folders:
    # Extraemos el Vendor y la Zona del nombre de la carpeta (ej. "0_262")
    # folder es algo como "Series_ID=0_262"
    id_str = folder.split("=")[1] 
    vendor_id, zone_id = id_str.split("_")
    
    # Leemos el parquet de esta combinación específica
    df_serie = pd.read_parquet(Path(base_path, folder))
    df_serie = df_serie.sort_values("timestamp")

    static_covs = pd.DataFrame({
        "VendorID": [float(vendor_id)],
        "PULocationID": [float(zone_id)]
    })
    
    ts_target = TimeSeries.from_dataframe(
        df_serie, 
        time_col="timestamp", 
        value_cols="demand",
        static_covariates=static_covs,
        freq="h",                      
        fill_missing_dates=True,      
        fillna_value=0.0,              
    )
    target_series_list.append(ts_target)
    
    # 2. Pasadas (Distancia y Precio)
    ts_past = TimeSeries.from_dataframe(
        df_serie, 
        time_col="timestamp", 
        value_cols=["avg_distance", "avg_amount"],
        freq="h",
        fill_missing_dates=True,
        fillna_value=0.0             
    )
    past_covariates_list.append(ts_past)
    
    # 3. Futuras (Tiempo trigonométrico)
    ts_future_raw = TimeSeries.from_dataframe(
        df_serie, 
        time_col="timestamp", 
        value_cols=["hour_sin", "hour_cos", "dow_sin", "dow_cos"],
        freq="h",
        fill_missing_dates=True
    )
    
    # Interpolamos los NaN para que la curva trigonométrica siga siendo perfecta
    filler = MissingValuesFiller()
    ts_future = filler.transform(ts_future_raw)
    future_covariates_list.append(ts_future)

print(f"Listas creadas con {len(target_series_list)} series temporales únicas.")

Listas creadas con 1042 series temporales únicas.


Vamos a crear los conjuntos de entrenamiento

In [12]:
MIN_LENGTH = 300  # Horas mínimas necesarias para que la red neuronal pueda aprender algo
corte_temporal = 0.80

#Vamos a seleccionar solo los datos a partir de 2023, que es el punto en el que se estabiliza la demanda
fecha_inicio = pd.Timestamp("2023-01-01")

train_target, val_target = [], []
train_past, val_past = [], []
train_future, val_future = [], []

series_descartadas = 0

for ts_t, ts_p, ts_f in zip(target_series_list, past_covariates_list, future_covariates_list):
    
    if ts_t.end_time() < fecha_inicio:
        series_descartadas += 1
        continue

    if ts_t.start_time() < fecha_inicio:
        ts_t2 = ts_t.drop_before(fecha_inicio)
        ts_p2 = ts_p.drop_before(fecha_inicio)
        ts_f2 = ts_f.drop_before(fecha_inicio)

    if len(ts_t2) < MIN_LENGTH:
        series_descartadas += 1
        continue

        
    # Calculamos la FECHA EXACTA del 80%
    punto_de_corte = ts_t2.get_timestamp_at_point(corte_temporal)
    
    # Cortamos las 3 series
    t_train, t_val = ts_t2.split_before(punto_de_corte)
    p_train, p_val = ts_p2.split_before(punto_de_corte)
    f_train, f_val = ts_f2.split_before(punto_de_corte)
    
    # Comprobación de seguridad 
    assert t_train.end_time() == p_train.end_time() == f_train.end_time(), "¡Error! Desalineación detectada."
    
    # Guardamos en las listas
    train_target.append(t_train)
    val_target.append(t_val)
    
    train_past.append(p_train)
    val_past.append(p_val)
    
    train_future.append(f_train)
    val_future.append(f_val)

print(f"¡Series divididas!")
print(f"Series válidas para entrenar: {len(train_target)}")
print(f"Series descartadas por ser muy cortas: {series_descartadas}")

¡Series divididas!
Series válidas para entrenar: 1037
Series descartadas por ser muy cortas: 5


hay zonas como se ve en el codigo que son descartadas por no tener precticamente historial, básicamente son outliers. Como el tide no puede predecir para estas zonas, simplemente relegaremos sus predicciones al otro modelo o a una baseline y ya

## ENTRENAMIENTO

In [14]:
%load_ext tensorboard
%tensorboard --logdir mis_logs_tide

In [23]:


# ==========================================
# FASE 1: ESCALADO DE DATOS
# ==========================================
print("Escalando los datos...")

# Escalamos la demanda (Target)
scaler_target = Scaler()
train_target_scaled = scaler_target.fit_transform(train_target)
val_target_scaled = scaler_target.transform(val_target)

# Escalamos la distancia y el precio (Past Covariates)
scaler_past = Scaler()
train_past_scaled = scaler_past.fit_transform(train_past)
val_past_scaled = scaler_past.transform(val_past)

# ==========================================
# FASE 2: CREACIÓN DEL MODELO TiDE
# ==========================================
print("Inicializando el cerebro TiDE...")

modelo_tide = TiDEModel(
    input_chunk_length=72,   
    output_chunk_length=24,  
    num_encoder_layers=3,  
    num_decoder_layers=3,  
    decoder_output_dim=32,
    hidden_size=256,
    
    likelihood=QuantileRegression(quantiles=[0.10, 0.50, 0.90]),

    batch_size=1024,          
    
    n_epochs=10,              
    optimizer_kwargs={"lr": 1e-3}, 
    random_state=42,         
    log_tensorboard=True,         
    work_dir="mis_logs_tide",
    
   
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": -1,
        "gradient_clip_val": 1.0
    }
)

#Pasamos los datos escalado a float 32 para poder usar la configuración 16 mixed de la GPU
print("Convirtiendo datos a Float32 para la GPU...")

train_target_scaled = [ts.astype(np.float32) for ts in train_target_scaled]
train_past_scaled = [ts.astype(np.float32) for ts in train_past_scaled]
train_future = [ts.astype(np.float32) for ts in train_future]

val_target_scaled = [ts.astype(np.float32) for ts in val_target_scaled]
val_past_scaled = [ts.astype(np.float32) for ts in val_past_scaled]
val_future = [ts.astype(np.float32) for ts in val_future]

# ==========================================
# FASE 3: ¡ENTRENAMIENTO!
# ==========================================
print("¡Arrancando el entrenamiento! (Paciencia, esto va a hacer sudar a tu máquina...)")

modelo_tide.fit(
    series=train_target_scaled,
    past_covariates=train_past_scaled,
    future_covariates=train_future,       # Recuerda: Estas NO se escalan
    val_series=val_target_scaled,
    val_past_covariates=val_past_scaled,
    val_future_covariates=val_future,
    dataloader_kwargs={"num_workers": 8},
    verbose=True                          # Para ver la barra de progreso
)

print("¡Entrenamiento finalizado con éxito!")

Escalando los datos...
Inicializando el cerebro TiDE...
Convirtiendo datos a Float32 para la GPU...
¡Arrancando el entrenamiento! (Paciencia, esto va a hacer sudar a tu máquina...)


number of `past_covariates` features is <= `temporal_width_past`, leading to feature expansion.number of covariates: 2, `temporal_width_past=4`.
number of `future_covariates` features is <= `temporal_width_future`, leading to feature expansion.number of covariates: 4, `temporal_width_future=4`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                  ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ criterion             │ MSELoss          │      0 │ eval  │     0 │
│ 1  │ train_criterion       │ MSELoss          │      0 │ eval  │     0 │
│ 2  │ val_criterion         │ MSELoss          │      0 │ eval  │     0 │
│ 3  │ train_metrics         │ MetricCollection │      0 │ train │     0 │
│ 4  │ val_metrics           │ MetricCollection │      0 │ train │     0 │
│ 5  │ past_cov_projection   │ _ResidualBlock   │  1.8 K │ train │     0 │
│ 6  │ future_cov_projection │ _ResidualBlock   │  2.3 K │ train │     0 │
│ 7  │ encoders              │ Sequential       │  843 K │ train │     0 │
│ 8  │ decoders              │ Sequential       │  1.6 M │ train │     0 │
│ 9  │ temporal_decoder      │ _ResidualBlock   │  3.6 K │ train │     0 │
│ 10 │ lookback_skip         │ Linear           │  5.3 K │ train │     0 │
└────┴───────────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 2.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.5 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 68                                                                                          
Modules in eval mode: 3                                                                                            
Total FLOPs: 0

Output()

/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21:
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:534: 
Found 3 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If 
this is intentional, you can ignore this warning.


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
